# 02 — TF-IDF + Logistic Regression Baseline

We build the **traditional NLP baseline** and evaluate it on the held-out
test split.

The pipeline has two stages:

1. `TfidfVectorizer` converts text to a sparse matrix of weighted n-gram
   frequencies. *Why*: text is high-dimensional and most words are uninformative.
   TF-IDF up-weights words that are frequent in *this* document but rare across
   the corpus, which yields a strong sparse feature representation.
2. `LogisticRegression(class_weight='balanced')` learns a linear decision
   boundary. *Why*: linear models on sparse text features are well-understood,
   fast, and interpretable via per-feature coefficients.

The expected number on the held-out test split is ≈ 0.88 accuracy — strong for
a baseline. (See §5 of `final_report.md` for the analysis of why this baseline
is so strong: *label-lexicon leakage*.)


In [1]:
# Path-setup boilerplate so the notebook can import src.*
import sys
from pathlib import Path
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
print("project root:", ROOT)


project root: /Users/spandarayamajhi/Desktop/Artificial Intelligence Coursework (Final Submission)


In [2]:
import json, numpy as np, pandas as pd
from sklearn.model_selection import train_test_split

from src import config
from src.models import traditional_ml
from src.evaluation import compute_metrics
from src.explainability import top_features_per_class


## 1. Load weak-labelled data

In [3]:
df = pd.read_csv(config.PROCESSED_CSV)
df['y'] = df['binary_label'].map(config.LABEL2ID)
print(f'rows={len(df):,}; positive rate={df.y.mean():.3f}')
df.head(2)


rows=14,961; positive rate=0.429


,campaign_id,category,text,binary_label,fine_label,y
0,10yr-old-kalicia-to-fight-cancer,Medical,10-year-old Kalicia is the firstborn of the Fr...,non_manipulative,non_manipulative,0
1,1230-miles-and-48-hours-chasing-a-cure,Medical,Hello! My name is Amy Rossiter Crist. I have a...,non_manipulative,non_manipulative,0


## 2. Train/test split (stratified, seed 42)

In [4]:
X = np.asarray(df['text'].astype(str).tolist(), dtype=object)
y = np.asarray(df['y'].tolist(), dtype=np.int64)
Xtr, Xte, ytr, yte = train_test_split(
    X, y, test_size=config.TRAIN_TEST_SPLIT,
    stratify=y, random_state=config.RANDOM_SEED)
print(f'train={len(Xtr):,}  test={len(Xte):,}')


train=11,968  test=2,993


## 3. Fit the pipeline

In [5]:
pipe, ypred, yproba = traditional_ml.fit_predict(Xtr, ytr, Xte)
print('vocabulary size:', len(pipe.named_steps['tfidf'].vocabulary_))


/Users/spandarayamajhi/Library/Python/3.14/lib/python/site-packages/sklearn/linear_model/_logistic.py:1184: FutureWarning: 'n_jobs' has no effect since 1.8 and will be removed in 1.10. You provided 'n_jobs=-1', please leave it unspecified.
  warnings.warn(msg, category=FutureWarning)


vocabulary size: 15000


## 4. Evaluate

In [6]:
m = compute_metrics(yte, ypred, yproba, 'TF-IDF + LR',
                    labels=config.LABELS_BINARY)
print(f'accuracy   {m.accuracy:.4f}')
print(f'precision  {m.precision:.4f}')
print(f'recall     {m.recall:.4f}')
print(f'f1         {m.f1:.4f}')
print(f'macro-f1   {m.macro_f1:.4f}')
print(f'ROC-AUC    {m.roc_auc:.4f}')
print(m.per_class_report)


accuracy   0.8794
precision  0.8817
recall     0.8302
f1         0.8552
macro-f1   0.8759
ROC-AUC    0.9485
                  precision    recall  f1-score   support

non_manipulative       0.88      0.92      0.90      1709
    manipulative       0.88      0.83      0.86      1284

        accuracy                           0.88      2993
       macro avg       0.88      0.87      0.88      2993
    weighted avg       0.88      0.88      0.88      2993



**Reading the confusion matrix.** Rows are true labels (`non_manipulative`,
`manipulative`), columns are predictions. We expect to see most errors as
*false negatives* — the model is precision-biased on the manipulative class.


In [7]:
from src import visualizations as viz
viz.plot_confusion_matrix(np.array(m.confusion),
                          labels=config.LABELS_BINARY,
                          model_name='TF-IDF + LR (notebook run)')


PosixPath('/Users/spandarayamajhi/Desktop/Artificial Intelligence Coursework (Final Submission)/outputs/figures/fig_confusion_tf-idf_+_lr_(notebook_run).png')

## 5. Top features (explainability)

In [8]:
top = top_features_per_class(pipe, n=20)
top


,manipulative_top,manipulative_weight,non_manipulative_top,non_manipulative_weight
0,funeral,21.1072,to help,-1.3907
1,please help,17.7108,bills,-1.3712
2,right now,12.6039,medical expenses,-1.3327
3,immediately,9.1863,please donate,-1.2258
4,death,8.6054,consider,-1.1901
5,right,8.2127,is currently,-1.1645
6,funeral expenses,7.5815,fire,-1.1576
7,please,7.0269,please consider,-1.1560
8,the funeral,5.4689,recovery,-1.1421
9,heartbreaking,5.0594,can help,-0.9969


Top manipulative features (`funeral`, `please help`, `right now`, `immediately`,
`death`, `funeral expenses`, `heartbreaking`, `asap`, `urgent`) are exactly the
n-grams the weak-label lexicons match. This is precisely the **label-lexicon
leakage** discussed in the final report.


## 6. Save artefacts

In [9]:
traditional_ml.save(pipe)
print('model saved at', config.MODELS_DIR / 'tfidf_lr.joblib')


model saved at /Users/spandarayamajhi/Desktop/Artificial Intelligence Coursework (Final Submission)/outputs/models/tfidf_lr.joblib
